In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

block_size = 64
batch_size = 128
max_iters = 3000
learning_rate = 3e-4
eval_iters = 100
eval_interval  = 500
n_embd = 384
n_layer = 8
n_head = 8
dropout = 0.2 # so the model doesn't overfit

cuda


In [2]:
chars = ""
with open('/kaggle/input/datasets/liannakadadi/wizard-of-oz/wizard_of_oz.txt'
,'r',encoding='utf-8') as f: # f as file name
    text = f.read()
    chars = sorted(list(set(text)))

vocab_size = len(chars)
print(len(text))

232316


In [3]:
#  tokenizer -> encoder: converts each elt to integer and decoder: converts back to characters

string_to_int = {ch:i for i, ch in enumerate(chars)}
int_to_string = { i:ch for i,ch in enumerate(chars)}
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

print(encode('hello'))

[61, 58, 65, 65, 68]


In [4]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data[:100])

tensor([80,  1,  1, 28, 39, 42, 39, 44, 32, 49,  1, 25, 38, 28,  1, 44, 32, 29,
         1, 47, 33, 50, 25, 42, 28,  1, 33, 38,  1, 39, 50,  0,  0,  1,  1, 26,
        49,  0,  0,  1,  1, 36, 11,  1, 30, 42, 25, 38, 35,  1, 26, 25, 45, 37,
         0,  0,  1,  1, 25, 45, 44, 32, 39, 42,  1, 39, 30,  1, 44, 32, 29,  1,
        47, 33, 50, 25, 42, 28,  1, 39, 30,  1, 39, 50,  9,  1, 44, 32, 29,  1,
        36, 25, 38, 28,  1, 39, 30,  1, 39, 50])


In [5]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
  data = train_data if split == 'train' else val_data
  ix = torch.randint(len(data) - block_size, (batch_size,))
  # print(ix)
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix]) # offset by one
  x, y = x.to(device), y.to(device)
  return x, y


In [6]:
# validation and training splits
# bigram -> 2 words, The probability of a word depends only on the previous word.
# block size and batch size
block_size = 8
x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print('when input is', context, 'target is', target)

when input is tensor([80]) target is tensor(1)
when input is tensor([80,  1]) target is tensor(1)
when input is tensor([80,  1,  1]) target is tensor(28)
when input is tensor([80,  1,  1, 28]) target is tensor(39)
when input is tensor([80,  1,  1, 28, 39]) target is tensor(42)
when input is tensor([80,  1,  1, 28, 39, 42]) target is tensor(39)
when input is tensor([80,  1,  1, 28, 39, 42, 39]) target is tensor(44)
when input is tensor([80,  1,  1, 28, 39, 42, 39, 44]) target is tensor(32)


In [7]:
@torch.no_grad()
def estimate_loss():
  out = {}
  model.eval()
  for split in ['train', 'val']:
    losses = torch.zeros(eval_iters)
    for k in range(eval_iters):
      X, Y = get_batch(split)
      logits, loss = model(X, Y)
      losses[k] = loss.item()
    out[split] = losses.mean()
  model.train()
  return out

In [8]:
class Head(nn.Module):
  """one head of self-attention"""

  def __init__(self, head_size):
    super().__init__()
    self.key = nn.Linear(n_embd, head_size, bias=False)
    self.query = nn.Linear(n_embd, head_size, bias=False)
    self.value = nn.Linear(n_embd, head_size, bias=False)
    self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    self.dropout = nn.Dropout(dropout)


  def forward(self, x):
    #  input of size (batch, time_step, channels)
    #  output of size (batch, time-step, head size)
    B,T,C = x.shape
    k= self.key(x) #(B,T, hs)
    q = self.query(x) #(B,T, hs)
    #  compute attention scores("affinities")
    wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5 #(B,T, hs) @ (B, hs, T) -> (B,T,T)
    wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B,T,T)
    wei = F.softmax(wei, dim=-1)  #(B,T,T)
    wei = self.dropout(wei)
    #  perform the weighted aggregation of the values
    v = self.value(x) #(B,T,hs)
    out = wei @ v # (B,T,T) @ (B,T, hs) -> (B,T,hs)
    return out


class MultiHeadAttention(nn.Module):
  """multiple heads of self-attention in parallel"""

  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
    self.proj = nn.Linear(head_size* num_heads, n_embd)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    out = torch.cat([h(x) for  h in self.heads], dim=-1) #(B, T, F)-> (B, T, [h1, h1, h1, h1, h2, h2, h2, h2, h3, h3, h3, h3])
    out = self.dropout(self.proj(out))
    return out


class FeedForward(nn.Module):
  """ a simple linear layer followed by a non-linearity"""

  def __init__(self, n_embd):
    super().__init__()
    self.net = nn.Sequential(
        #  Linear -> ReLU -> linear
        nn.Linear(n_embd, 4*n_embd),
        nn.ReLU(),
        nn.Linear(4* n_embd, n_embd),
        nn.Dropout(dropout),
    )

  def forward(self, x):
    return self.net(x)


class Block(nn.Module):
  """ Transformer block: communication followed by computation"""

  def __init__(self, n_embd, n_head):
    # n_embd: embedding dimensions, n_head: the number of heads we'd like
    super().__init__()
    head_size = n_embd // n_head
    self.sa = MultiHeadAttention(n_head, head_size)
    self.ffwd = FeedForward(n_embd)
    self.ln1 = nn.LayerNorm(n_embd)
    self.ln2 = nn.LayerNorm(n_embd)

# post norm
  def forward(self, x):
    y = self.sa(x)
    x = self.ln1(x+y)
    y = self.ffwd(x)
    x = self.ln2(x+y)
    return x

class GPTLanguageModel(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
    self.position_embedding_table = nn.Embedding(block_size, n_embd)
    self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
    self.ln_f = nn.LayerNorm(n_embd) # final layer norm
    self.lm_head = nn.Linear(n_embd, vocab_size)

    self.apply(self.__init__weights)


  def __init__weights(self, module):
    if isinstance(module, nn.Linear):
      torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
      if module.bias is not None:
        torch.nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
      torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

  def forward(self, index, targets=None):
    B, T = index.shape
    logits = self.token_embedding_table(index)

    # index and targets are both (B, T) tensor of integers
    tok_emb = self.token_embedding_table(index) # (B, T, C)
    pos_emb = self.position_embedding_table(torch.arange(T, device= device)) #(T, C)
    x = tok_emb + pos_emb # (B, T, C)
    x = self.blocks(x) #(B, T, C)
    x = self.ln_f(x) #(B, T, C) final layer norm
    logits = self.lm_head(x) #(B, T, vocab_size)


    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      logits = logits.view(B*T, C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)

    return logits, loss

    # if targets is None:
    #   loss = None
    # else:
    #   B,T,C = logits.shape
    #   logits = logits.view(B*T, C)
    #   tarets = targets.view(B*T)
    #   loss =F.cross_entropy(logits, targets)

    # return logits, loss

  def generate(self, index, max_new_tokens):
    #  index is (B,T) array of indices in the current context
    for _ in range(max_new_tokens):
      #  get the predictions
      logits, loss = self.forward(index)
      # focus only on the last time step
      logits = logits[:, -1, :]# becomes(b,C)
      #  apply softmax to get probabilities
      probs = F.softmax(logits, dim=-1) #(B,C)
      # sample from the distribution
      index_next = torch.multinomial(probs, num_samples=1) #(B, 1)
      # append sampled index to the running sequence
      index = torch.cat((index, index_next), dim=1) #(B, T+1)
    return index


model = GPTLanguageModel(vocab_size)
m = model.to(device)

#  context = torch.zeros((1,1), dtype = torch.long, device=device)
# generated_chars = decode(m.genearte(context, max_new_tokens=500)[0].tolist())
# print(generated_chars)


In [9]:
import torch

# create optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

start_iter = 0

for iter in range(start_iter, max_iters):

    # evaluate loss occasionally
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.3f}, val loss: {losses['val']:.3f}")

    # sample batch
    xb, yb = get_batch('train')

    # forward pass
    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    # save checkpoint every 500 iterations
    if iter % 500 == 0:
        torch.save({
            'iteration': iter,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': loss.item()
        }, 'checkpoint.pth')

        print(f"Checkpoint saved at iteration {iter}")

print(loss.item())


step: 0, train loss: 4.472, val loss: 4.473
Checkpoint saved at iteration 0
step: 100, train loss: 2.217, val loss: 2.290
step: 200, train loss: 2.014, val loss: 2.107
step: 300, train loss: 1.904, val loss: 2.012
step: 400, train loss: 1.839, val loss: 1.962
step: 500, train loss: 1.794, val loss: 1.918
Checkpoint saved at iteration 500
step: 600, train loss: 1.745, val loss: 1.870
step: 700, train loss: 1.733, val loss: 1.863
step: 800, train loss: 1.693, val loss: 1.853
step: 900, train loss: 1.682, val loss: 1.829
step: 1000, train loss: 1.665, val loss: 1.827
Checkpoint saved at iteration 1000
step: 1100, train loss: 1.643, val loss: 1.811
step: 1200, train loss: 1.639, val loss: 1.795
step: 1300, train loss: 1.616, val loss: 1.789
step: 1400, train loss: 1.618, val loss: 1.798
step: 1500, train loss: 1.605, val loss: 1.771
Checkpoint saved at iteration 1500
step: 1600, train loss: 1.609, val loss: 1.781
step: 1700, train loss: 1.588, val loss: 1.765
step: 1800, train loss: 1.582,

In [10]:
vocab = set()

input_file = "/kaggle/input/datasets/liannakadadi/wizard-of-oz/wizard_of_oz.txt"
output_file_train = "output_train.txt"
output_file_val = "output_val.txt"
vocab_file = "vocab.txt"

with open(input_file, "r", encoding="utf-8") as f:
    text = f.read()

# Split 90% train, 10% validation
split_index = int(len(text) * 0.9)

train_text = text[:split_index]
val_text = text[split_index:]

# Save training data
with open(output_file_train, "w", encoding="utf-8") as f:
    f.write(train_text)

# Save validation data
with open(output_file_val, "w", encoding="utf-8") as f:
    f.write(val_text)

# Build vocabulary
vocab.update(set(text))

# Save vocabulary
with open(vocab_file, "w", encoding="utf-8") as f:
    for char in sorted(vocab):
        f.write(char + "\n")

print("Processing complete!")


Processing complete!


In [11]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import mmap
import random 

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

block_size = 64
batch_size = 128
max_iters = 3000
learning_rate = 3e-4
eval_iters = 100
eval_interval  = 500
n_embd = 384
n_layer = 8
n_head = 8
dropout = 0.2 # so the model doesn't overfit

cuda


In [12]:
chars = ""
with open('/kaggle/working/vocab.txt'
,'r',encoding='utf-8') as f: # f as file name
    text = f.read()
    chars = sorted(list(set(text)))

vocab_size = len(chars)
print(len(text))

162


In [13]:
#  tokenizer -> encoder: converts each elt to integer and decoder: converts back to characters

string_to_int = {ch:i for i, ch in enumerate(chars)}
int_to_string = { i:ch for i,ch in enumerate(chars)}
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])


In [16]:
import os
os.listdir("/kaggle/working")


['vocab.txt', '.virtual_documents', 'val.txt', 'checkpoint.pth', 'train.txt']

In [15]:
import os
#  to rename the files in the /kaggle/working directory
os.rename("/kaggle/working/output_train.txt",
          "/kaggle/working/train.txt")

os.rename("/kaggle/working/output_val.txt",
          "/kaggle/working/val.txt")

In [17]:
#  create a mapping from characters to integers
stoi = { ch:i for i, ch in enumerate(chars)}
itos = { i:ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# memory map for using small snippets of text from a single file of any size
def get_random_chunk(split):
    filename = "/kaggle/working/train.txt" if split == 'train' else "/kaggle/working/val.txt"
    with open(filename, 'rb') as f:
        with mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ) as mm:
            # Determine the file size and a random position to start reading
            file_size = len(mm)
            start_pos = random.randint(0, (file_size) - block_size*batch_size)

            # Seek to the random position and read the block of text
            mm.seek(start_pos)
            block = mm.read(block_size*batch_size-1)
            
            # Decode the block to a string, ignoring any invalid byte sequences
            decoded_block = block.decode('utf-8', errors='ignore').replace('\r','')
            
            # Train and test splits
            data = torch.tensor(encode(decoded_block), dtype=torch.long)
    
    return data

def get_batch(split):
    data = get_random_chunk(split)
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y
            

In [18]:
@torch.no_grad()
def estimate_loss():
  out = {}
  model.eval()
  for split in ['train', 'val']:
    losses = torch.zeros(eval_iters)
    for k in range(eval_iters):
      X, Y = get_batch(split)
      logits, loss = model(X, Y)
      losses[k] = loss.item()
    out[split] = losses.mean()
  model.train()
  return out

In [19]:
class Head(nn.Module):
  """one head of self-attention"""

  def __init__(self, head_size):
    super().__init__()
    self.key = nn.Linear(n_embd, head_size, bias=False)
    self.query = nn.Linear(n_embd, head_size, bias=False)
    self.value = nn.Linear(n_embd, head_size, bias=False)
    self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    self.dropout = nn.Dropout(dropout)


  def forward(self, x):
    #  input of size (batch, time_step, channels)
    #  output of size (batch, time-step, head size)
    B,T,C = x.shape
    k= self.key(x) #(B,T, hs)
    q = self.query(x) #(B,T, hs)
    #  compute attention scores("affinities")
    wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5 #(B,T, hs) @ (B, hs, T) -> (B,T,T)
    wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B,T,T)
    wei = F.softmax(wei, dim=-1)  #(B,T,T)
    wei = self.dropout(wei)
    #  perform the weighted aggregation of the values
    v = self.value(x) #(B,T,hs)
    out = wei @ v # (B,T,T) @ (B,T, hs) -> (B,T,hs)
    return out


class MultiHeadAttention(nn.Module):
  """multiple heads of self-attention in parallel"""

  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
    self.proj = nn.Linear(head_size* num_heads, n_embd)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    out = torch.cat([h(x) for  h in self.heads], dim=-1) #(B, T, F)-> (B, T, [h1, h1, h1, h1, h2, h2, h2, h2, h3, h3, h3, h3])
    out = self.dropout(self.proj(out))
    return out


class FeedForward(nn.Module):
  """ a simple linear layer followed by a non-linearity"""

  def __init__(self, n_embd):
    super().__init__()
    self.net = nn.Sequential(
        #  Linear -> ReLU -> linear
        nn.Linear(n_embd, 4*n_embd),
        nn.ReLU(),
        nn.Linear(4* n_embd, n_embd),
        nn.Dropout(dropout),
    )

  def forward(self, x):
    return self.net(x)


class Block(nn.Module):
  """ Transformer block: communication followed by computation"""

  def __init__(self, n_embd, n_head):
    # n_embd: embedding dimensions, n_head: the number of heads we'd like
    super().__init__()
    head_size = n_embd // n_head
    self.sa = MultiHeadAttention(n_head, head_size)
    self.ffwd = FeedForward(n_embd)
    self.ln1 = nn.LayerNorm(n_embd)
    self.ln2 = nn.LayerNorm(n_embd)

# post norm
  def forward(self, x):
    y = self.sa(x)
    x = self.ln1(x+y)
    y = self.ffwd(x)
    x = self.ln2(x+y)
    return x

class GPTLanguageModel(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
    self.position_embedding_table = nn.Embedding(block_size, n_embd)
    self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
    self.ln_f = nn.LayerNorm(n_embd) # final layer norm
    self.lm_head = nn.Linear(n_embd, vocab_size)

    self.apply(self.__init__weights)


  def __init__weights(self, module):
    if isinstance(module, nn.Linear):
      torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
      if module.bias is not None:
        torch.nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
      torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

  def forward(self, index, targets=None):
    B, T = index.shape
    logits = self.token_embedding_table(index)

    # index and targets are both (B, T) tensor of integers
    tok_emb = self.token_embedding_table(index) # (B, T, C)
    pos_emb = self.position_embedding_table(torch.arange(T, device= device)) #(T, C)
    x = tok_emb + pos_emb # (B, T, C)
    x = self.blocks(x) #(B, T, C)
    x = self.ln_f(x) #(B, T, C) final layer norm
    logits = self.lm_head(x) #(B, T, vocab_size)


    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      logits = logits.view(B*T, C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)

    return logits, loss

    # if targets is None:
    #   loss = None
    # else:
    #   B,T,C = logits.shape
    #   logits = logits.view(B*T, C)
    #   tarets = targets.view(B*T)
    #   loss =F.cross_entropy(logits, targets)

    # return logits, loss

  def generate(self, index, max_new_tokens):
    #  index is (B,T) array of indices in the current context
    for _ in range(max_new_tokens):
      #  get the predictions
      logits, loss = self.forward(index)
      # focus only on the last time step
      logits = logits[:, -1, :]# becomes(b,C)
      #  apply softmax to get probabilities
      probs = F.softmax(logits, dim=-1) #(B,C)
      # sample from the distribution
      index_next = torch.multinomial(probs, num_samples=1) #(B, 1)
      # append sampled index to the running sequence
      index = torch.cat((index, index_next), dim=1) #(B, T+1)
    return index


model = GPTLanguageModel(vocab_size)
m = model.to(device)

#  context = torch.zeros((1,1), dtype = torch.long, device=device)
# generated_chars = decode(m.genearte(context, max_new_tokens=500)[0].tolist())
# print(generated_chars)


In [21]:
import torch

# create optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

start_iter = 0

for iter in range(start_iter, max_iters):

    # evaluate loss occasionally
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.3f}, val loss: {losses['val']:.3f}")

    # sample batch
    xb, yb = get_batch('train')

    # forward pass
    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    # save checkpoint every 500 iterations
    if iter % 500 == 0:
        torch.save({
            'iteration': iter,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': loss.item()
        }, 'checkpoint.pth')

        print(f"Checkpoint saved at iteration {iter}")

print(loss.item())


step: 0, train loss: 0.796, val loss: 1.542
Checkpoint saved at iteration 0
step: 100, train loss: 0.767, val loss: 1.588
step: 200, train loss: 0.719, val loss: 1.615
step: 300, train loss: 0.699, val loss: 1.613
step: 400, train loss: 0.661, val loss: 1.680
step: 500, train loss: 0.636, val loss: 1.682
Checkpoint saved at iteration 500
step: 600, train loss: 0.604, val loss: 1.721
step: 700, train loss: 0.581, val loss: 1.744
step: 800, train loss: 0.574, val loss: 1.773
step: 900, train loss: 0.534, val loss: 1.794
step: 1000, train loss: 0.505, val loss: 1.857
Checkpoint saved at iteration 1000
step: 1100, train loss: 0.477, val loss: 1.865
step: 1200, train loss: 0.470, val loss: 1.879
step: 1300, train loss: 0.447, val loss: 1.923
step: 1400, train loss: 0.416, val loss: 1.927
step: 1500, train loss: 0.405, val loss: 2.011
Checkpoint saved at iteration 1500
step: 1600, train loss: 0.397, val loss: 1.991
step: 1700, train loss: 0.366, val loss: 2.044
step: 1800, train loss: 0.358,

In [22]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import mmap
import random 
import pickle 

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

# block_size = 32
# batch_size = 128
batch_size = 64
block_size = 128
max_iters = 200
learning_rate = 3e-4
eval_iters = 100
# eval_interval  = 500
n_embd = 384
n_layer = 1
n_head = 1
dropout = 0.2 # so the model doesn't overfit

cuda


In [23]:
chars = ""
with open('/kaggle/working/vocab.txt','r',encoding='utf-8') as f: # f as file name
    text = f.read()
    chars = sorted(list(set(text)))

vocab_size = len(chars)
print(len(text))

162


In [26]:
#  tokenizer -> encoder: converts each elt to integer and decoder: converts back to characters

string_to_int = {ch:i for i, ch in enumerate(chars)}
int_to_string = { i:ch for i,ch in enumerate(chars)}
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

In [27]:
# memory map for using small snippets of text from a single file of any size
def get_random_chunk(split):
    filename = "/kaggle/working/train.txt" if split == 'train' else "/kaggle/working/val.txt"
    with open(filename, 'rb') as f:
        with mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ) as mm:
            # Determine the file size and a random position to start reading
            file_size = len(mm)
            start_pos = random.randint(0, (file_size) - block_size*batch_size)

            # Seek to the random position and read the block of text
            mm.seek(start_pos)
            block = mm.read(block_size*batch_size-1)
            
            # Decode the block to a string, ignoring any invalid byte sequences
            decoded_block = block.decode('utf-8', errors='ignore').replace('\r','')
            
            # Train and test splits
            data = torch.tensor(encode(decoded_block), dtype=torch.long)
    
    return data

def get_batch(split):
    data = get_random_chunk(split)
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [28]:
@torch.no_grad()
def estimate_loss():
  out = {}
  model.eval()
  for split in ['train', 'val']:
    losses = torch.zeros(eval_iters)
    for k in range(eval_iters):
      X, Y = get_batch(split)
      logits, loss = model(X, Y)
      losses[k] = loss.item()
    out[split] = losses.mean()
  model.train()
  return out

In [32]:
class Head(nn.Module):
  """one head of self-attention"""

  def __init__(self, head_size):
    super().__init__()
    self.key = nn.Linear(n_embd, head_size, bias=False)
    self.query = nn.Linear(n_embd, head_size, bias=False)
    self.value = nn.Linear(n_embd, head_size, bias=False)
    self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    self.dropout = nn.Dropout(dropout)


  def forward(self, x):
    #  input of size (batch, time_step, channels)
    #  output of size (batch, time-step, head size)
    B,T,C = x.shape
    k= self.key(x) #(B,T, hs)
    q = self.query(x) #(B,T, hs)
    #  compute attention scores("affinities")
    wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5 #(B,T, hs) @ (B, hs, T) -> (B,T,T)
    wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B,T,T)
    wei = F.softmax(wei, dim=-1)  #(B,T,T)
    wei = self.dropout(wei)
    #  perform the weighted aggregation of the values
    v = self.value(x) #(B,T,hs)
    out = wei @ v # (B,T,T) @ (B,T, hs) -> (B,T,hs)
    return out


class MultiHeadAttention(nn.Module):
  """multiple heads of self-attention in parallel"""

  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
    self.proj = nn.Linear(head_size* num_heads, n_embd)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    out = torch.cat([h(x) for  h in self.heads], dim=-1) #(B, T, F)-> (B, T, [h1, h1, h1, h1, h2, h2, h2, h2, h3, h3, h3, h3])
    out = self.dropout(self.proj(out))
    return out


class FeedForward(nn.Module):
  """ a simple linear layer followed by a non-linearity"""

  def __init__(self, n_embd):
    super().__init__()
    self.net = nn.Sequential(
        #  Linear -> ReLU -> linear
        nn.Linear(n_embd, 4*n_embd),
        nn.ReLU(),
        nn.Linear(4* n_embd, n_embd),
        nn.Dropout(dropout),
    )

  def forward(self, x):
    return self.net(x)


class Block(nn.Module):
  """ Transformer block: communication followed by computation"""

  def __init__(self, n_embd, n_head):
    # n_embd: embedding dimensions, n_head: the number of heads we'd like
    super().__init__()
    head_size = n_embd // n_head
    self.sa = MultiHeadAttention(n_head, head_size)
    self.ffwd = FeedForward(n_embd)
    self.ln1 = nn.LayerNorm(n_embd)
    self.ln2 = nn.LayerNorm(n_embd)

# post norm
  def forward(self, x):
    y = self.sa(x)
    x = self.ln1(x+y)
    y = self.ffwd(x)
    x = self.ln2(x+y)
    return x

class GPTLanguageModel(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
    self.position_embedding_table = nn.Embedding(block_size, n_embd)
    self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
    self.ln_f = nn.LayerNorm(n_embd) # final layer norm
    self.lm_head = nn.Linear(n_embd, vocab_size)

    self.apply(self.__init__weights)


  def __init__weights(self, module):
    if isinstance(module, nn.Linear):
      torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
      if module.bias is not None:
        torch.nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
      torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

  def forward(self, index, targets=None):
    B, T = index.shape
    logits = self.token_embedding_table(index)

    # index and targets are both (B, T) tensor of integers
    tok_emb = self.token_embedding_table(index) # (B, T, C)
    pos_emb = self.position_embedding_table(torch.arange(T, device= device)) #(T, C)
    x = tok_emb + pos_emb # (B, T, C)
    x = self.blocks(x) #(B, T, C)
    x = self.ln_f(x) #(B, T, C) final layer norm
    logits = self.lm_head(x) #(B, T, vocab_size)


    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      logits = logits.view(B*T, C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)

    return logits, loss

    # if targets is None:
    #   loss = None
    # else:
    #   B,T,C = logits.shape
    #   logits = logits.view(B*T, C)
    #   tarets = targets.view(B*T)
    #   loss =F.cross_entropy(logits, targets)

    # return logits, loss

  def generate(self, index, max_new_tokens):
    #  index is (B,T) array of indices in the current context
    for _ in range(max_new_tokens):
      #  get the predictions
      logits, loss = self.forward(index)
      # focus only on the last time step
      logits = logits[:, -1, :]# becomes(b,C)
      #  apply softmax to get probabilities
      probs = F.softmax(logits, dim=-1) #(B,C)
      # sample from the distribution
      index_next = torch.multinomial(probs, num_samples=1) #(B, 1)
      # append sampled index to the running sequence
      index = torch.cat((index, index_next), dim=1) #(B, T+1)
    return index


model = GPTLanguageModel(vocab_size)
print('loading model parameters...')
with open('model-01.pkl', 'rb') as f:
    model = pickle.load(f)
print('loaded successfully')
m = model.to(device)

#  context = torch.zeros((1,1), dtype = torch.long, device=device)
# generated_chars = decode(m.genearte(context, max_new_tokens=500)[0].tolist())
# print(generated_chars)


loading model parameters...
loaded successfully


In [31]:
import torch

# create optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

start_iter = 0

for iter in range(start_iter, max_iters):

    # evaluate loss occasionally
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.3f}, val loss: {losses['val']:.3f}")

    # sample batch
    xb, yb = get_batch('train')

    # forward pass
    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    # save checkpoint every 500 iterations
    if iter % 500 == 0:
        torch.save({
            'iteration': iter,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': loss.item()
        }, 'checkpoint.pth')

        print(f"Checkpoint saved at iteration {iter}")

print(loss.item())

with open('model-01.pkl', 'wb') as f:
    pickle.dump(model, f)
print('model saved')

step: 0, train loss: 4.427, val loss: 4.432
Checkpoint saved at iteration 0
step: 100, train loss: 2.453, val loss: 2.527
2.3840444087982178
model saved


In [33]:
%%writefile training.py

import torch
import torch.nn as nn
from torch.nn import functional as F
import mmap
import random 
import pickle 
import argparse

parser = argparse.ArgumentParser(description='This is a demonstration program')

# Here we add an argument to the parser, specifying the expected type, a help message, etc.
parser.add_argument('-batch_size', type=str, required=True, help='Please provide a batch_size')

args = parser.parse_args()

# Now we can use the argument value in our program
print(f'batch size: {args.batch_size}')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

block_size = 128
batch_size = int(args.batch_size)

max_iters = 200
learning_rate = 3e-4
eval_iters = 100
# eval_interval  = 500
n_embd = 384
n_layer = 1
n_head = 1
dropout = 0.2 # so the model doesn't overfit


chars = ""
with open('/kaggle/working/vocab.txt','r',encoding='utf-8') as f: # f as file name
    text = f.read()
    chars = sorted(list(set(text)))

vocab_size = len(chars)
print(len(text))


#  tokenizer -> encoder: converts each elt to integer and decoder: converts back to characters

string_to_int = {ch:i for i, ch in enumerate(chars)}
int_to_string = { i:ch for i,ch in enumerate(chars)}
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])


# memory map for using small snippets of text from a single file of any size
def get_random_chunk(split):
    filename = "/kaggle/working/train.txt" if split == 'train' else "/kaggle/working/val.txt"
    with open(filename, 'rb') as f:
        with mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ) as mm:
            # Determine the file size and a random position to start reading
            file_size = len(mm)
            start_pos = random.randint(0, (file_size) - block_size*batch_size)

            # Seek to the random position and read the block of text
            mm.seek(start_pos)
            block = mm.read(block_size*batch_size-1)
            
            # Decode the block to a string, ignoring any invalid byte sequences
            decoded_block = block.decode('utf-8', errors='ignore').replace('\r','')
            
            # Train and test splits
            data = torch.tensor(encode(decoded_block), dtype=torch.long)
    
    return data

def get_batch(split):
    data = get_random_chunk(split)
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
  out = {}
  model.eval()
  for split in ['train', 'val']:
    losses = torch.zeros(eval_iters)
    for k in range(eval_iters):
      X, Y = get_batch(split)
      logits, loss = model(X, Y)
      losses[k] = loss.item()
    out[split] = losses.mean()
  model.train()
  return out

import torch


class Head(nn.Module):
  """one head of self-attention"""

  def __init__(self, head_size):
    super().__init__()
    self.key = nn.Linear(n_embd, head_size, bias=False)
    self.query = nn.Linear(n_embd, head_size, bias=False)
    self.value = nn.Linear(n_embd, head_size, bias=False)
    self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    self.dropout = nn.Dropout(dropout)


  def forward(self, x):
    #  input of size (batch, time_step, channels)
    #  output of size (batch, time-step, head size)
    B,T,C = x.shape
    k= self.key(x) #(B,T, hs)
    q = self.query(x) #(B,T, hs)
    #  compute attention scores("affinities")
    wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5 #(B,T, hs) @ (B, hs, T) -> (B,T,T)
    wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B,T,T)
    wei = F.softmax(wei, dim=-1)  #(B,T,T)
    wei = self.dropout(wei)
    #  perform the weighted aggregation of the values
    v = self.value(x) #(B,T,hs)
    out = wei @ v # (B,T,T) @ (B,T, hs) -> (B,T,hs)
    return out


class MultiHeadAttention(nn.Module):
  """multiple heads of self-attention in parallel"""

  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
    self.proj = nn.Linear(head_size* num_heads, n_embd)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    out = torch.cat([h(x) for  h in self.heads], dim=-1) #(B, T, F)-> (B, T, [h1, h1, h1, h1, h2, h2, h2, h2, h3, h3, h3, h3])
    out = self.dropout(self.proj(out))
    return out


class FeedForward(nn.Module):
  """ a simple linear layer followed by a non-linearity"""

  def __init__(self, n_embd):
    super().__init__()
    self.net = nn.Sequential(
        #  Linear -> ReLU -> linear
        nn.Linear(n_embd, 4*n_embd),
        nn.ReLU(),
        nn.Linear(4* n_embd, n_embd),
        nn.Dropout(dropout),
    )

  def forward(self, x):
    return self.net(x)


class Block(nn.Module):
  """ Transformer block: communication followed by computation"""

  def __init__(self, n_embd, n_head):
    # n_embd: embedding dimensions, n_head: the number of heads we'd like
    super().__init__()
    head_size = n_embd // n_head
    self.sa = MultiHeadAttention(n_head, head_size)
    self.ffwd = FeedForward(n_embd)
    self.ln1 = nn.LayerNorm(n_embd)
    self.ln2 = nn.LayerNorm(n_embd)

# post norm
  def forward(self, x):
    y = self.sa(x)
    x = self.ln1(x+y)
    y = self.ffwd(x)
    x = self.ln2(x+y)
    return x

class GPTLanguageModel(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
    self.position_embedding_table = nn.Embedding(block_size, n_embd)
    self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
    self.ln_f = nn.LayerNorm(n_embd) # final layer norm
    self.lm_head = nn.Linear(n_embd, vocab_size)

    self.apply(self.__init__weights)


  def __init__weights(self, module):
    if isinstance(module, nn.Linear):
      torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
      if module.bias is not None:
        torch.nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
      torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

  def forward(self, index, targets=None):
    B, T = index.shape
    logits = self.token_embedding_table(index)

    # index and targets are both (B, T) tensor of integers
    tok_emb = self.token_embedding_table(index) # (B, T, C)
    pos_emb = self.position_embedding_table(torch.arange(T, device= device)) #(T, C)
    x = tok_emb + pos_emb # (B, T, C)
    x = self.blocks(x) #(B, T, C)
    x = self.ln_f(x) #(B, T, C) final layer norm
    logits = self.lm_head(x) #(B, T, vocab_size)


    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      logits = logits.view(B*T, C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)

    return logits, loss

    # if targets is None:
    #   loss = None
    # else:
    #   B,T,C = logits.shape
    #   logits = logits.view(B*T, C)
    #   tarets = targets.view(B*T)
    #   loss =F.cross_entropy(logits, targets)

    # return logits, loss

  def generate(self, index, max_new_tokens):
    #  index is (B,T) array of indices in the current context
    for _ in range(max_new_tokens):
      #  get the predictions
      logits, loss = self.forward(index)
      # focus only on the last time step
      logits = logits[:, -1, :]# becomes(b,C)
      #  apply softmax to get probabilities
      probs = F.softmax(logits, dim=-1) #(B,C)
      # sample from the distribution
      index_next = torch.multinomial(probs, num_samples=1) #(B, 1)
      # append sampled index to the running sequence
      index = torch.cat((index, index_next), dim=1) #(B, T+1)
    return index


model = GPTLanguageModel(vocab_size)
# print('loading model parameters...')
# with open('model-01.pkl', 'rb') as f:
#     model = pickle.load(f)
# print('loaded successfully')
m = model.to(device)



# create optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

start_iter = 0

for iter in range(start_iter, max_iters):

    # evaluate loss occasionally
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.3f}, val loss: {losses['val']:.3f}")

    # sample batch
    xb, yb = get_batch('train')

    # forward pass
    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    # save checkpoint every 500 iterations
    if iter % 500 == 0:
        torch.save({
            'iteration': iter,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': loss.item()
        }, 'checkpoint.pth')

        print(f"Checkpoint saved at iteration {iter}")

print(loss.item())

# with open('model-01.pkl', 'wb') as f:
#     pickle.dump(model, f)
# print('model saved')

torch.save(model.state_dict(), '/kaggle/working/model-01.pkl')
print('model saved')

Writing training.py


In [ ]:
# %%writefile chatbot.py

# import torch
# import torch.nn as nn
# from torch.nn import functional as F
# import mmap
# import random 
# import pickle 
# import argparse

# parser = argparse.ArgumentParser(description='This is a demonstration program')

# # Here we add an argument to the parser, specifying the expected type, a help message, etc.
# parser.add_argument('-batch_size', type=int, required=True, help='Please provide a batch_size')

# args = parser.parse_args()

# # Now we can use the argument value in our program
# print(f'batch size: {args.batch_size}')

# device = 'cuda' if torch.cuda.is_available() else 'cpu'
# print(device)

# block_size = 128
# batch_size = args.batch_size

# max_iters = 200
# learning_rate = 3e-4
# eval_iters = 100
# # eval_interval  = 500
# n_embd = 384
# n_layer = 1
# n_head = 1
# dropout = 0.2 # so the model doesn't overfit

# chars = ""
# with open('/kaggle/working/vocab.txt','r',encoding='utf-8') as f: # f as file name
#     text = f.read()
#     chars = sorted(list(set(text)))

# vocab_size = len(chars)
# print(len(text))

# #  tokenizer -> encoder: converts each elt to integer and decoder: converts back to characters

# string_to_int = {ch:i for i, ch in enumerate(chars)}
# int_to_string = { i:ch for i,ch in enumerate(chars)}
# encode = lambda s: [string_to_int[c] for c in s]
# decode = lambda l: ''.join([int_to_string[i] for i in l])


# class Head(nn.Module):
#   """one head of self-attention"""

#   def __init__(self, head_size):
#     super().__init__()
#     self.key = nn.Linear(n_embd, head_size, bias=False)
#     self.query = nn.Linear(n_embd, head_size, bias=False)
#     self.value = nn.Linear(n_embd, head_size, bias=False)
#     self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

#     self.dropout = nn.Dropout(dropout)


#   def forward(self, x):
#     #  input of size (batch, time_step, channels)
#     #  output of size (batch, time-step, head size)
#     B,T,C = x.shape
#     k= self.key(x) #(B,T, hs)
#     q = self.query(x) #(B,T, hs)
#     #  compute attention scores("affinities")
#     wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5 #(B,T, hs) @ (B, hs, T) -> (B,T,T)
#     wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B,T,T)
#     wei = F.softmax(wei, dim=-1)  #(B,T,T)
#     wei = self.dropout(wei)
#     #  perform the weighted aggregation of the values
#     v = self.value(x) #(B,T,hs)
#     out = wei @ v # (B,T,T) @ (B,T, hs) -> (B,T,hs)
#     return out


# class MultiHeadAttention(nn.Module):
#   """multiple heads of self-attention in parallel"""

#   def __init__(self, num_heads, head_size):
#     super().__init__()
#     self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
#     self.proj = nn.Linear(head_size* num_heads, n_embd)
#     self.dropout = nn.Dropout(dropout)

#   def forward(self, x):
#     out = torch.cat([h(x) for  h in self.heads], dim=-1) #(B, T, F)-> (B, T, [h1, h1, h1, h1, h2, h2, h2, h2, h3, h3, h3, h3])
#     out = self.dropout(self.proj(out))
#     return out


# class FeedForward(nn.Module):
#   """ a simple linear layer followed by a non-linearity"""

#   def __init__(self, n_embd):
#     super().__init__()
#     self.net = nn.Sequential(
#         #  Linear -> ReLU -> linear
#         nn.Linear(n_embd, 4*n_embd),
#         nn.ReLU(),
#         nn.Linear(4* n_embd, n_embd),
#         nn.Dropout(dropout),
#     )

#   def forward(self, x):
#     return self.net(x)


# class Block(nn.Module):
#   """ Transformer block: communication followed by computation"""

#   def __init__(self, n_embd, n_head):
#     # n_embd: embedding dimensions, n_head: the number of heads we'd like
#     super().__init__()
#     head_size = n_embd // n_head
#     self.sa = MultiHeadAttention(n_head, head_size)
#     self.ffwd = FeedForward(n_embd)
#     self.ln1 = nn.LayerNorm(n_embd)
#     self.ln2 = nn.LayerNorm(n_embd)

# # post norm
#   def forward(self, x):
#     y = self.sa(x)
#     x = self.ln1(x+y)
#     y = self.ffwd(x)
#     x = self.ln2(x+y)
#     return x

# class GPTLanguageModel(nn.Module):
#   def __init__(self, vocab_size):
#     super().__init__()
#     self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
#     self.position_embedding_table = nn.Embedding(block_size, n_embd)
#     self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
#     self.ln_f = nn.LayerNorm(n_embd) # final layer norm
#     self.lm_head = nn.Linear(n_embd, vocab_size)

#     self.apply(self.__init__weights)


#   def __init__weights(self, module):
#     if isinstance(module, nn.Linear):
#       torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
#       if module.bias is not None:
#         torch.nn.init.zeros_(module.bias)
#     elif isinstance(module, nn.Embedding):
#       torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

#   def forward(self, index, targets=None):
#     B, T = index.shape
#     # logits = self.token_embedding_table(index)

#     # index and targets are both (B, T) tensor of integers
#     tok_emb = self.token_embedding_table(index) # (B, T, C)
#     pos_emb = self.position_embedding_table(torch.arange(T, device= device)) #(T, C)
#     x = tok_emb + pos_emb # (B, T, C)
#     x = self.blocks(x) #(B, T, C)
#     x = self.ln_f(x) #(B, T, C) final layer norm
#     logits = self.lm_head(x) #(B, T, vocab_size)


#     if targets is None:
#       loss = None
#     else:
#       B, T, C = logits.shape
#       logits = logits.view(B*T, C)
#       targets = targets.view(B*T)
#       loss = F.cross_entropy(logits, targets)

#     return logits, loss

#     # if targets is None:
#     #   loss = None
#     # else:
#     #   B,T,C = logits.shape
#     #   logits = logits.view(B*T, C)
#     #   tarets = targets.view(B*T)
#     #   loss =F.cross_entropy(logits, targets)

#     # return logits, loss

    
#   # def generate(self, index, max_new_tokens):
#   #   #  index is (B,T) array of indices in the current context
#   #   for _ in range(max_new_tokens):
#   #     # crop idx to the last block_size tokens
#   #     index_cond = index[:, -block_size:]
#   #     #  get the predictions
#   #     logits, loss = self.forward(index_cond)
#   #     # focus only on the last time step
#   #     logits = logits[:, -1, :]# becomes(b,C)
#   #     #  apply softmax to get probabilities
#   #     probs = F.softmax(logits, dim=-1) #(B,C)
#   #     # sample from the distribution
#   #     index_next = torch.multinomial(probs, num_samples=1) #(B, 1)
#   #     # append sampled index to the running sequence
#   #     index = torch.cat((index, index_next), dim=1) #(B, T+1)
#   #   return index

# def generate(self, idx, max_new_tokens):
    
#     for _ in range(max_new_tokens):
    
#         idx_cond = idx[:, -block_size:]   # FIX
    
#         logits, loss = self.forward(idx_cond)
    
#         logits = logits[:, -1, :]
#         probs = F.softmax(logits, dim=-1)
    
#         idx_next = torch.multinomial(probs, num_samples=1)
    
#         idx = torch.cat((idx, idx_next), dim=1)
    
#     return idx


# model = GPTLanguageModel(vocab_size)
# print('loading model parameters...')
# # with open('/kaggle/working/model-01.pkl', 'rb') as f:
# #     model = pickle.load(f)
# model.load_state_dict(torch.load('/kaggle/working/model-01.pkl', map_location=device))
# print('loaded successfully')
# m = model.to(device)
# m.eval()

# #  context = torch.zeros((1,1), dtype = torch.long, device=device)
# # generated_chars = decode(m.genearte(context, max_new_tokens=500)[0].tolist())
# # print(generated_chars)


# prompt = "Hello"

# context = torch.tensor(encode(prompt), dtype=torch.long, device=device)

# generated_chars = decode(
#     m.generate(context.unsqueeze(0), max_new_tokens=150)[0].tolist()
# )

# print("Completion:\n", generated_chars)

# # while True:
#     # prompt = input("Prompt:\n")
#     # context = torch.tensor(encode(prompt), dtype=torch.long, device=device)
#     # generated_chars = decode(m.generate(context.unsqueeze(0), max_new_tokens = 150)[0].tolist())
#     # print(f'Completion:\n{generated_chars}')
  

In [34]:
%%writefile chatbot.py

import torch
import torch.nn as nn
from torch.nn import functional as F
import argparse

parser = argparse.ArgumentParser(description='Chatbot using GPT-like model')
parser.add_argument('-batch_size', type=int, required=True, help='Please provide a batch_size')
args = parser.parse_args()

print(f'batch size: {args.batch_size}')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

# ----------------- hyperparameters -----------------
block_size = 128
batch_size = args.batch_size
n_embd = 384
n_layer = 1
n_head = 1
dropout = 0.2

# ----------------- load vocab -----------------
with open('/kaggle/working/vocab.txt', 'r', encoding='utf-8') as f:
    text = f.read()
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"Text length: {len(text)}, Vocab size: {vocab_size}")

string_to_int = {ch:i for i,ch in enumerate(chars)}
int_to_string = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

# ----------------- model components -----------------
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size*num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd),
            nn.ReLU(),
            nn.Linear(4*n_embd, n_embd),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        y = self.sa(x)
        x = self.ln1(x+y)
        y = self.ffwd(x)
        x = self.ln2(x+y)
        return x

# ----------------- GPT model -----------------
class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self.__init_weights)

    def __init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, index, targets=None):
        B, T = index.shape
        tok_emb = self.token_embedding_table(index)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    # ✅ generate method must be inside the class
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self.forward(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# ----------------- load model -----------------
model = GPTLanguageModel(vocab_size)
print('loading model parameters...')
model.load_state_dict(torch.load('/kaggle/working/model-01.pkl', map_location=device))
print('loaded successfully')
model = model.to(device)
model.eval()

# ----------------- generate text -----------------
prompt = "Hello"
context = torch.tensor(encode(prompt), dtype=torch.long, device=device)
generated_chars = decode(model.generate(context.unsqueeze(0), max_new_tokens=150)[0].tolist())
print("Completion:\n", generated_chars)


Writing chatbot.py


In [35]:
!python training.py -batch_size 32


batch size: 32
cuda
162
step: 0, train loss: 4.434, val loss: 4.443
Checkpoint saved at iteration 0
step: 100, train loss: 2.477, val loss: 2.538
2.40836763381958
model saved


In [36]:
!python chatbot.py -batch_size 32

batch size: 32
cuda
Text length: 162, Vocab size: 81
loading model parameters...
loaded successfully
Completion:
 Hellous os canlane tofplondis.
cargh tothecorissid Sowanglle wooon ted becope fe t t st sepethare
s, andhizassat uaid per be im fastotre s licalonend ghowe


In [ ]:
#  time and efficiency testing

%%writefile time-testing.py 
import time 
start_time = time.time()
for i in range(10000):
    print(i*2)

end_time = time.time()
total_time = end_time - start_time
print(f'time taken: {total_time}')

In [ ]:
!python time-testing.py

In [ ]:
!ls /kaggle/working


In [ ]:
import importlib
import training
import chatbot

importlib.reload(training)
importlib.reload(chatbot)


In [ ]:
model = training.train_model(
    model,
    optimizer,
    get_batch,
    estimate_loss,
    max_iters,
    eval_iters
)
